In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
# Colab already provides pyarrow, pandas, numpy, plotly.
# Only duckdb and polars need to be installed.
!pip install duckdb==1.4.4 polars==1.40.0 --quiet

In [ ]:
# ── Cell 2: Mount Google Drive and configure paths ─────────────────────────
# This triggers an OAuth browser prompt the first time per session.
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import shutil

# ── USER CONFIGURATION ─────────────────────────────────────────────────────
# Adjust DRIVE_BASE if you placed b3_data/ somewhere other than MyDrive root.
DRIVE_BASE   = Path("/content/drive/MyDrive/b3_data")
DRIVE_RAW    = DRIVE_BASE / "raw"      # COTAHIST TXT/ZIP files live here
DRIVE_DB_DIR = DRIVE_BASE / "db"       # DuckDB file is synced here

LOCAL_DB     = Path("/content/b3_data.duckdb")   # ephemeral workspace

# Create Drive folders if they don't exist yet
DRIVE_RAW.mkdir(parents=True, exist_ok=True)
DRIVE_DB_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_DB = DRIVE_DB_DIR / "b3_data.duckdb"

# ── Copy DB from Drive to local (or start fresh) ───────────────────────────
if DRIVE_DB.exists():
    print(f"Copying DB from Drive → {LOCAL_DB} ...")
    shutil.copy2(DRIVE_DB, LOCAL_DB)
    size_mb = LOCAL_DB.stat().st_size / 1_048_576
    print(f"Done. DB size: {size_mb:.1f} MB")
else:
    print("No existing DB found on Drive. A new one will be created during ingest.")

# ── IMPORTANT: DuckDB file lock warning ────────────────────────────────────
print(
    "\n⚠  WARNING: Do NOT open b3_data.duckdb from another machine or Colab "
    "session while this session is active. DuckDB holds an exclusive file lock."
)

In [ ]:
# ── Cell 3: Run ingest ─────────────────────────────────────────────────────
# Adds the project scripts/ directory to the Python path so run_ingest
# can be imported without installing the package.
import sys

# Adjust this path if the notebook is not inside the project repo structure.
# If running from a Drive copy, update accordingly:
SCRIPTS_DIR = Path("/content/drive/MyDrive/b3_data/scripts")
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from ingest import run_ingest

# Safe pattern: ingest + sync in try/finally
try:
    run_ingest(data_dir=DRIVE_RAW, db_path=LOCAL_DB)
finally:
    if LOCAL_DB.exists():
        shutil.copy2(LOCAL_DB, DRIVE_DB)
        size_mb = DRIVE_DB.stat().st_size / 1_048_576
        print(f"\nDB synced to Drive. Size: {size_mb:.1f} MB")

In [ ]:
# ── Cell 4: Quick verification ─────────────────────────────────────────────
import duckdb

con = duckdb.connect(str(LOCAL_DB), read_only=True)

print("=== Ingested files ===")
display(con.sql("SELECT filename, ingested_at FROM ingested_files ORDER BY ingested_at").df())

print("\n=== Row count per year ===")
display(con.sql("""
    SELECT YEAR(datpre) AS ano, COUNT(*) AS registros
    FROM cotacoes
    GROUP BY 1
    ORDER BY 1
""").df())

print("\n=== Latest 5 trading dates ===")
display(con.sql("""
    SELECT DISTINCT datpre
    FROM cotacoes
    ORDER BY datpre DESC
    LIMIT 5
""").df())

con.close()

In [ ]:
# ── Cell 5: Sync DB back to Drive (standalone safety cell) ─────────────────
# Run this cell even if Cell 3 failed (or run it standalone before closing
# the session). Ensures the DB is synced regardless of upstream errors.

try:
    if LOCAL_DB.exists():
        print(f"Copying DB → Drive ({DRIVE_DB}) ...")
        shutil.copy2(LOCAL_DB, DRIVE_DB)
        size_mb = DRIVE_DB.stat().st_size / 1_048_576
        print(f"Done. DB size on Drive: {size_mb:.1f} MB")
    else:
        print("No local DB file found — nothing to copy back.")
except Exception as e:
    print(f"ERROR during copy-back: {e}")
    raise